In [0]:
%sql
SELECT
    month,
    day,
    COUNT(*) AS num_flights
FROM flights
GROUP BY month, day
ORDER BY month, day;

In [0]:
%sql
SELECT
    month,
    day,
    COUNT(*) AS num_flights,
    ROW_NUMBER() OVER (
        PARTITION BY month
        ORDER BY COUNT(*) DESC
    ) AS rank
FROM flights
GROUP BY month, day;

In [0]:
%sql
WITH ranked_days AS (
    SELECT
        month,
        day,
        COUNT(*) AS num_flights,
        ROW_NUMBER() OVER (
            PARTITION BY month
            ORDER BY COUNT(*) DESC
        ) AS rank
    FROM flights
    GROUP BY month, day
)
SELECT *
FROM ranked_days
WHERE rank = 1
ORDER BY month;

In [0]:
%sql
SELECT
    WEEKOFYEAR(MAKE_DATE(year, month, day)) AS week,
    ROUND(AVG(arr_delay), 2) AS avg_delay
FROM flights
WHERE arr_delay IS NOT NULL
GROUP BY WEEKOFYEAR(MAKE_DATE(year, month, day))
ORDER BY week;

In [0]:
%sql
WITH weekly_delays AS (
    SELECT
        WEEKOFYEAR(MAKE_DATE(year, month, day)) AS week,
        ROUND(AVG(arr_delay), 2) AS avg_delay
    FROM flights
    WHERE arr_delay IS NOT NULL
    GROUP BY WEEKOFYEAR(MAKE_DATE(year, month, day))
)

SELECT
    week,
    avg_delay,
    LAG(avg_delay) OVER (ORDER BY week) AS prev_week
FROM weekly_delays
ORDER BY week;

In [0]:
%sql
WITH weekly_delays AS (
    SELECT
        WEEKOFYEAR(MAKE_DATE(year, month, day)) AS week,
        ROUND(AVG(arr_delay), 2) AS avg_delay
    FROM flights
    WHERE arr_delay IS NOT NULL
    GROUP BY WEEKOFYEAR(MAKE_DATE(year, month, day))
)

SELECT
    week,
    avg_delay,
    LAG(avg_delay) OVER (ORDER BY week) AS prev_week,
    ROUND(
        avg_delay - LAG(avg_delay) OVER (ORDER BY week),
        2
    ) AS abs_change
FROM weekly_delays
ORDER BY week;

In [0]:
%sql
WITH weekly_delays AS (
    SELECT
        WEEKOFYEAR(MAKE_DATE(year, month, day)) AS week,
        ROUND(AVG(arr_delay), 2) AS avg_delay
    FROM flights
    WHERE arr_delay IS NOT NULL
    GROUP BY WEEKOFYEAR(MAKE_DATE(year, month, day))
)

SELECT
    week,
    avg_delay,
    LAG(avg_delay) OVER (ORDER BY week) AS prev_week,

    -- absolute change
    ROUND(
        avg_delay - LAG(avg_delay) OVER (ORDER BY week),
        2
    ) AS abs_change,

    -- percentage change
    ROUND(
        100 * (avg_delay - LAG(avg_delay) OVER (ORDER BY week)) /
        LAG(avg_delay) OVER (ORDER BY week),
        2
    ) AS pct_change

FROM weekly_delays
ORDER BY week;

In [0]:
%sql
SELECT
    carrier,
    month,
    COUNT(*) AS total_flights,
    SUM(CASE WHEN arr_delay <= 0 THEN 1 ELSE 0 END) AS on_time_flights
FROM flights
WHERE arr_delay IS NOT NULL
GROUP BY carrier, month
ORDER BY month;

In [0]:
%sql
SELECT
    carrier,
    month,

    ROUND(
        100.0 * SUM(CASE WHEN arr_delay <= 0 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS on_time_pct

FROM flights
WHERE arr_delay IS NOT NULL
GROUP BY carrier, month
ORDER BY month;

In [0]:
%sql
WITH stats AS (
    SELECT
        carrier,
        month,
        ROUND(
            100.0 * SUM(CASE WHEN arr_delay <= 0 THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS on_time_pct
    FROM flights
    WHERE arr_delay IS NOT NULL
    GROUP BY carrier, month
)

SELECT
    carrier,
    month,
    on_time_pct,

    DENSE_RANK() OVER (
        PARTITION BY month
        ORDER BY on_time_pct DESC
    ) AS rank

FROM stats
ORDER BY month, rank;

In [0]:
%sql
SELECT
    carrier,
    month,
    flight,
    arr_delay,

    ROUND(
        AVG(arr_delay) OVER (PARTITION BY carrier, month),
        2
    ) AS carrier_month_avg

FROM flights
WHERE arr_delay IS NOT NULL
LIMIT 20;

In [0]:
%sql
SELECT
    carrier,
    month,
    flight,
    arr_delay,

    ROUND(
        AVG(arr_delay) OVER (PARTITION BY carrier, month),
        2
    ) AS carrier_month_avg,

    ROUND(
        arr_delay - AVG(arr_delay) OVER (PARTITION BY carrier, month),
        2
    ) AS vs_avg

FROM flights
WHERE arr_delay IS NOT NULL
ORDER BY vs_avg DESC
LIMIT 20;